# *This notebook provides a workflow to run BfBio, including*:
- Generating the input network
- Determining the optimal damping factor value
- Producing Personalized PageRank (PPR) predictions
- Computing and plotting random permutations (as described in the original paper)
- Retrieving the best ranked genes based on the threshold determined with the permutations
- Preparing a subfolder dedicated to running a PubMed search

# <span style = "color:pink">**Importing the required libraries**</span>

In [1]:
import sys
sys.path.append("../scripts")

import glob
import io
import json
import omics_analysis
import os
import math
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import re
import seaborn as sns
import shutil
import statistics
import string_analysis
import sys

from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score
from sklearn.model_selection import KFold
from tqdm import tqdm

# <span style = "color:blue">**Reading the training genes and setting the input parameters**</span>

In [ ]:
# Setting the process of interest
process = "stalk_cell"

# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()
print(f"There are {len(genes)} training genes in this list")

# Setting the process of interest
process = "stalk_cell"

# Setting the dataset considered
dataset = "STRING_CS_100_E-GEOD-45750_corr_07"

# <span style = "color:brown">**Computing a benchmark on a dataset to find the best damping factor value**</span>

In [ ]:
# Setting the path to the graph
dataset = "GeneCards_E-GEOD-45750_corr_07" 
path_graph = f"../graphs/integrated" 

# Create a folder for the upcoming threshold benchmark
path_benchmark = f"../results/{process}"
path_benchmark_dataset = f"{path_benchmark}/{dataset}"
path_results = f"{path_benchmark_dataset}/Benchmark"

if not os.path.exists(f"{path_benchmark}"):
    os.mkdir(f"{path_benchmark}")

if not os.path.exists(f"{path_benchmark_dataset}"):
    os.mkdir(f"{path_benchmark_dataset}")

if not os.path.exists(f"{path_results}"):
    os.mkdir(f"{path_results}")
else:
    shutil.rmtree(f"{path_results}")
    os.mkdir(f"{path_results}")

# Reading the graph file 
G = nx.read_graphml(f"{path_graph}/graph_{dataset}.graphml")

# Running a benchmark to determine the best damping factor
benchmark_results = useful_functions.analyse(G, genes)
benchmark_results.to_csv(f"{path_results}/PageRank_evaluation_{dataset}_{process}.csv",
        sep = ",", index = False)

# Retrieving the best DF and saving the file as hyperparameters
hyperparameters_df = benchmark_results[benchmark_results["auc_roc"] == benchmark_results["auc_roc"].max()]
hyperparameters_df.to_csv(f"{path_graph}/Hyperparameters_{dataset}_{process}.csv", sep = ",", index = False)

# <span style = "color:green">**Running a Personalized PageRank on a graph to predict genes**</span>

### *Setting the input parameters*

In [ ]:
# Setting path graph and optimal damping factor value
path_graph = f"../graphs/integrated/graph_{dataset}.graphml"
df = 0.1 # optimal damping factor value determined previously

# Setting the output directory
if not os.path.exists("../results"):
    os.mkdir("../results")

if not os.path.exists(f"../results/{process}"):
    os.mkdir(f"../results.{process}")

if not os.path.exists(f"../results/{process}/{dataset}"):
    os.mkdir(f"../results/{process}/{dataset}")
    
PPR_dir = f"../results/{process}/{dataset}/PPR"
if not os.path.exists(PPR_dir):
    os.mkdir(PPR_dir)

# Creating folders to store the permutations results
permut_dir = f"{PPR_dir}/Permutations"
if not os.path.exists(f"{permut_dir}"):
    os.mkdir(f"{permut_dir}")
    os.mkdir(f"{permut_dir}/Perm_graph_perm_genes")
    os.mkdir(f"{permut_dir}/Perm_graph_real_genes")
    os.mkdir(f"{permut_dir}/Real_graph_perm_genes")
    os.mkdir(f"{permut_dir}/Real_data")

# Reading the .graphml file
G = nx.read_graphml(path_graph)
print("The graph has been generated !")
print(f"There are {len(G.nodes())} genes and {len(G.edges())} edges in this graph")

# Extracting valid genes
valid_genes = [node for node in G.nodes(data = False) if node in genes]
#valid_genes = [data["name"] for _, data in G.nodes(data = True) if data["name"] in genes] 
print(f"There are {len(valid_genes)} valid genes in this graph")

# Saving these results in a .txt file
with open(f"../results/{process}/{dataset}/graph_information.txt", "w") as f_out:
    f_out.write(f"Number of genes in the graph: {len(G.nodes())}\n")
    f_out.write(f"Number of edges in the graph: {len(G.edges())}\n")
    f_out.write(f"Number of training genes in the graph: {len(valid_genes)}")

### *Generating the PPR predictions and the permutations results*

In [ ]:
# Running a 5 fold-CV to assess the performance of the PageRank algorithm on the dataset selected
os.mkdir(f"{PPR_dir}/5_folds")

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
folds = list(kf.split(valid_genes))
k = 1000

fold_evaluations = []
AUCs = []
predicted_rank = []
i = 1
for fold_idx, (train_idx, test_idx) in enumerate(folds):
    fold = fold_idx + 1
    train_genes = [valid_genes[i] for i in train_idx]  # Training set (seed nodes)
    test_genes = [valid_genes[i] for i in test_idx]    # Test set
    
    # Build the personalization vector (1 for training genes, 0 for others)
    personalization = {node: 0 for node in G.nodes}
    for node  in G.nodes(data=False):
        if node in train_genes:
            personalization[node] = 1.0
    
    # Normalize the personalization vector
    total = sum(personalization.values())
    personalization = {k: v / total for k, v in personalization.items()}
    
    # Run Personalized PageRank
    ppr_scores = nx.pagerank(G, alpha=df, personalization=personalization , weight="weight")

    ppr_scores_filtered = {
        gene: score
        for gene, score in ppr_scores.items()
        if gene not in train_genes # Ensure the node not in the training genes
    }
    model = pd.DataFrame({"Gene": ppr_scores_filtered.keys(),
                     "PageRank_score": ppr_scores_filtered.values()})
    model["Label"] = 0
    model.loc[model.Gene.isin(test_genes), "Label"] = 1
    
    model.to_csv(f"{PPR_dir}/5_folds/Model_{i}.csv", sep = ",", index = False)

    model["rank"] = list(range(1, len(model) + 1))
    rank_left_out = model.loc[model.Gene.isin(test_genes), "rank"].tolist()
    mean_value = np.mean(rank_left_out)
    predicted_rank.append(mean_value)

    X = model[["PageRank_score"]]
    y = model["Label"]
    auc = round(metrics.roc_auc_score(y, X), 4)
    AUCs.append(auc)

    with io.open(f"{PPR_dir}/5_folds/Summary_average_rank_models.txt", "a", encoding = 'utf-8') as f_out:
        f_out.write("************\nModel {}\n- Training genes left out : {}\n- Average predicted rank : {:.0f} out of {}\n- AUC : {:.3f}\n"
                    .format(i, test_genes, mean_value, len(model["Gene"]), auc))
    i += 1

mean_rank_global = np.mean(predicted_rank)
AUC_global = np.mean(AUCs)

with io.open(f"{PPR_dir}/5_folds/Summary_average_rank_models.txt", "a", encoding = 'utf-8') as f_out:
    f_out.write("------------\nOverall, the mean predicted rank of a training gene with a 5-fold CV is {:.0f} out of {}\nThe overall AUC is {:.3f} ± {:.3f}"
                .format(mean_rank_global, len(model["Gene"]), AUC_global, statistics.stdev(AUCs)))

evaluation = useful_functions.evaluate_fold(test_genes, ppr_scores_filtered, k=k)
fold_evaluations.append({"fold": fold, **evaluation})
folds_evaluation = fold_evaluations
folds_evaluation = pd.DataFrame.from_dict(folds_evaluation)
folds_evaluation.to_csv(f"{PPR_dir}/5_fold-CV_{dataset}_{process}.csv", sep = ",", index = False)

# Averaging the results
mean_rank = folds_evaluation["mean_rank"].mean()
mean_median_rank = folds_evaluation["median_rank"].mean()
mean_roc_auc = folds_evaluation["auc_roc"].mean()

with open(f"{PPR_dir}/5_fold-CV_results.txt","w") as f_out:
    f_out.write(f"Mean rank: {mean_rank}\n")
    f_out.write(f"Mean median rank: {mean_median_rank}\n")
    f_out.write(f"Mean ROC-AUC: {mean_roc_auc}")

# Plotting the 5 fold-CV ROC curves
data_1 = pd.read_csv(f"{PPR_dir}/5_folds/Model_1.csv")
data_2 = pd.read_csv(f"{PPR_dir}/5_folds/Model_2.csv")
data_3 = pd.read_csv(f"{PPR_dir}/5_folds/Model_3.csv")
data_4 = pd.read_csv(f"{PPR_dir}/5_folds/Model_4.csv")
data_5 = pd.read_csv(f"{PPR_dir}/5_folds/Model_5.csv")

# define the predictor variables and the response variable for each model

X_1 = data_1[['PageRank_score']]
y_1 = data_1['Label']

X_2 = data_2[['PageRank_score']]
y_2 = data_2['Label']

X_3 = data_3[['PageRank_score']]
y_3 = data_3['Label']

X_4 = data_4[['PageRank_score']]
y_4 = data_4['Label']

X_5 = data_5[['PageRank_score']]
y_5 = data_5['Label']

# define metrics
fpr_1, tpr_1, _ = metrics.roc_curve(y_1,  X_1)
fpr_2, tpr_2, _ = metrics.roc_curve(y_2,  X_2)
fpr_3, tpr_3, _ = metrics.roc_curve(y_3,  X_3)
fpr_4, tpr_4, _ = metrics.roc_curve(y_4,  X_4)
fpr_5, tpr_5, _ = metrics.roc_curve(y_5,  X_5)

# create ROC curve
plt.figure(figsize=(10, 6))

plt.plot([0, 1], [0, 1], "k--", label="random classifier (AUC = 0.5)")
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')

auc_1 = round(metrics.roc_auc_score(y_1, X_1), 4)
auc_2 = round(metrics.roc_auc_score(y_2, X_2), 4)
auc_3 = round(metrics.roc_auc_score(y_3, X_3), 4)
auc_4 = round(metrics.roc_auc_score(y_4, X_4), 4)
auc_5 = round(metrics.roc_auc_score(y_5, X_5), 4)

aucs = [auc_1, auc_2, auc_3, auc_4, auc_5]

plt.plot(fpr_1,tpr_1, label = "AUC fold 1 = " +str(f"{auc_1:.3f}"), color = "red")
plt.plot(fpr_2,tpr_2, label = "AUC fold 2 = " +str(f"{auc_2:.3f}"), color = "blue")
plt.plot(fpr_3,tpr_3, label = "AUC fold 3 = " +str(f"{auc_3:.3f}"), color = "green")
plt.plot(fpr_4,tpr_4, label = "AUC fold 4 = " +str(f"{auc_4:.3f}"), color = "purple")
plt.plot(fpr_5,tpr_5, label = "AUC fold 5 = " +str(f"{auc_5:.3f}"), color = "brown")

plt.plot([], [], ' ', label = "mean AUC = " +str(f"{np.mean(aucs):.3f}") +str(f" ± {statistics.stdev(aucs):.3f}"))

# add legend
plt.legend()
plt.title(f"Roc curves - {dataset} PPR - {process} - 5 fold CV - Damping factor = {df}")

# saving the graph
plt.savefig(f"{PPR_dir}/ROC_curves_5_folds-CV_{dataset}_{process}.png")

# showing the graph
plt.show()

# Running the Personalized PageRank with the best DF to rank the genes in the network
predictions = useful_functions.predict_all(df, G, valid_genes)
predictions = predictions[predictions["Label"] == 0] #removing the training genes
predictions.to_csv(f"{PPR_dir}/PPR_predictions_{dataset}_{process}.csv",
                  sep = ",", index = False)

# Applying permutations to find the best threshold to analyze the prioritized genes
useful_functions.permutations_predict_test(process, df, permut_dir, G, valid_genes)

### *Plotting the Random vs Real data scores distribution and visually determining the optimal threshold*

By visualizing these distributions, a threshold designed to minimize the inclusion of random PageRank score distributions while maximizing the retention of high-scoring genes in the actual PageRank score distributions can be determined. Don't hesistate to run this cell as many times as needed to find the best threshold before moving on with the rest of the workflow.

In [ ]:
# Step 1: For each type of permutation, read the files and make an average in a column
# perm graph perm genes
files = glob.glob(f"{permut_dir}/Perm_graph_perm_genes/*.csv")
df_perm_graph_perm_genes = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_perm_graph_perm_genes.to_csv(f"{permut_dir}/perm_graph_perm_genes.csv", sep = ",", index = False)

# perm graph real genes
files = glob.glob(f"{permut_dir}/Perm_graph_real_genes/*.csv")
df_perm_graph_real_genes = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_perm_graph_real_genes.to_csv(f"{permut_dir}/perm_graph_real_genes.csv", sep = ",", index = False)

# real graph perm genes
files = glob.glob(f"{permut_dir}/Real_graph_perm_genes/*.csv")
df_real_graph_perm_genes = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_real_graph_perm_genes.to_csv(f"{permut_dir}/real_graph_perm_genes.csv", sep = ",", index = False)

# real data
files = glob.glob(f"{permut_dir}/Real_data/*.csv")
df_real_data = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_real_data.to_csv(f"{permut_dir}/real_data.csv", sep = ",", index = False)

# Step 2: Make a DataFrame with the real and permutated data
distributions = pd.DataFrame({"Real data": list(np.log10(df_real_data["Score"])),
                             "Permuted graph permuted genes": list(np.log10(df_perm_graph_perm_genes["Score"])),
                             "Permuted graph real genes": list(np.log10(df_perm_graph_real_genes["Score"])),
                             "Real graph permuted genes": list(np.log10(df_real_graph_perm_genes["Score"]))})

distributions.to_csv(f"{permut_dir}/log10_permutations_distributions.csv", sep = ",", index = False)

# Step 3: Plot the distribution curves
# Plot parameters
threshold = -4.2  #determined visually
sns.set_theme(rc = {'figure.figsize':(11.7, 8.27)})
sns.set_style('whitegrid')

for column in distributions.columns:
    sns.kdeplot(distributions[column], fill = True, label = column)

# Add a vertical line for the score threshold
threshold_line = plt.axvline(x = threshold, color = "red", linestyle = '--', label = f'Threshold ({threshold})')

plt.legend(fontsize = 15, loc = "upper left")
plt.title(f"Score distribution (log10 scale) of the Real vs Permuted data - {dataset} - {process} - Damping factor {df}")
plt.xlabel("Score", size = 20)
plt.ylabel("Density", size = 20)
plt.savefig(f"{permut_dir}/Distribution_real_data_vs_permuted_data.png")
plt.show()

### *Using the threshold to retrieve the predicted genes*

Using the threshold previously determined, the best ranked genes can be retrieved and their enrichment can be assessed using EnrichR's API.

In [ ]:
# Importing Seaborn again
sns.set_style("white")

# Converting the log10 threshold to the normal value
threshold_pr = math.pow(10, threshold)
print(f"The selected PR threshold is log10 = {threshold} ==> {threshold_pr}")

# Reading the PPR ranking file
ranking = pd.read_csv(f"{PPR_dir}/PPR_predictions_{dataset}_{process}.csv")

# Removing the training genes from the PPR file results
ranking = ranking[~ranking.Gene.isin(valid_genes)]

# Using the threshold to filter out the ranked genes
ranking_filtered = ranking[ranking['Score'] >= threshold_pr]
ranking_filtered.to_csv(f"{PPR_dir}/Top_{len(ranking_filtered)}_ranked_genes.csv",
                       sep = ",", index = False)

with open(f"{PPR_dir}/Top_{len(ranking_filtered)}_ranked_genes.txt", "w") as f_out:
    for i in ranking_filtered.itertuples():
        f_out.write(f"{i[1]}\n")

print(f"Before filtering: {ranking.shape}")
print(f"After filtering: {ranking_filtered.shape}")

# Assessing the enrichment of the best ranked genes
if not os.path.exists(f"{PPR_dir}/Enrichment"):
    os.mkdir(f"{PPR_dir}/Enrichment")
    os.mkdir(f"{PPR_dir}/Enrichment/Best_ranked_genes")
    os.mkdir(f"{PPR_dir}/Enrichment/Training_genes")

else:
    shutil.rmtree(f"{PPR_dir}/Enrichment")
    os.mkdir(f"{PPR_dir}/Enrichment")
    os.mkdir(f"{PPR_dir}/Enrichment/Best_ranked_genes")
    os.mkdir(f"{PPR_dir}/Enrichment/Training_genes")

# Enrichment parameters
enrichr_library = "GO_Biological_Process_2025"
color = "lightskyblue"
threshold_pathways = 10

if threshold_pathways == 10:
    x_param = 24
    y_param = 12

else:
    diff = threshold_pathways - 10
    x_param = 24 + (diff * 1.2)
    y_param = 12 + (diff * 0.6)

# Retrieving the relevant GO annotations for the training genes
results = useful_functions.Enrichr_API(valid_genes, [enrichr_library])
df2 = results[1]
df2.rename(columns = {0: "Index",
                     1: "Term",
                     2: "pvalues",
                     3: "Odds ratio",
                     4: "Combined score",
                     5: "Overlap genes",
                     6: "Adjusted pvalues",
                     7: "unknown1",
                     8: "unknown2"}, inplace = True)

df2 = df2.drop(columns = ["Index", "unknown1", "unknown2"])
df_relevant = df2.loc[df2["Adjusted pvalues"] < 0.05]
df_relevant.to_csv(f"{PPR_dir}/Enrichment/Training_genes/relevant_GO_training_genes.csv",
                  sep = ",", index = False)

# Generating the barplots for the top 10 enriched pathways
plot_name = f"{PPR_dir}/Enrichment/Training_genes/enrichment_top_{threshold_pathways}_pathways_training_genes.png"
results = useful_functions.Enrichr_API_top_n(genes, [enrichr_library], threshold_pathways)
useful_functions.enrichr_figure(results[2], results[3], results[4], plot_name, [enrichr_library], color, threshold_pathways, x_param, y_param)

# Retrieving the relevant GO annotations from the best ranked genes
df_top_ranked_genes = pd.read_csv(f"{PPR_dir}/Top_{len(ranking_filtered)}_ranked_genes.csv")
gene_list_input = [ ]

for gene in df_top_ranked_genes.itertuples():
    gene_list_input.append(gene[1].upper())

genes = [x.strip() for x in gene_list_input]
results = useful_functions.Enrichr_API(genes, [enrichr_library])

df2 = results[1]
df2.rename(columns = {0:'Index', 1: 'Term', 2: 'pvalues', 
                               3: 'Odds ratio', 4: 'Combined score', 
                               5: 'Overlap genes', 6: 'Adjusted pvalues', 
                               7: 'unknown1', 8: "unknown2"}, inplace = True)
    
df2 = df2.drop(columns = ["Index", "unknown1", "unknown2"])
df_relevant = df2.loc[df2["Adjusted pvalues"] < 0.05]
df_relevant.to_csv(f"{PPR_dir}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv",
                  sep = ",", index = False)

# Generating the barplots for the top 10 enriched pathways
plot_name = f"{PPR_dir}/Enrichment/Best_ranked_genes/enrichment_top_{threshold_pathways}_pathways_best_ranked_genes.png"
results = useful_functions.Enrichr_API_top_n(genes, [enrichr_library], threshold_pathways)
useful_functions.enrichr_figure(results[2], results[3], results[4], plot_name, [enrichr_library], color, threshold_pathways, x_param, y_param) 

# Looking for common relevant GO annotations
TG = []
df_TG = pd.read_csv(f"{PPR_dir}/Enrichment/Training_genes/relevant_GO_training_genes.csv")

for GO in df_TG.itertuples():
    TG.append(GO[1])
    
Best_genes = []
df_best_genes = pd.read_csv(f"{PPR_dir}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv")

for GO in df_best_genes.itertuples():
    Best_genes.append(GO[1])
    
common = []
for GO in Best_genes:
    if GO in TG:
        common.append(GO)
        
ratio = (len(common)/len(TG))*100

print(f"There are {len(common)} GO annotations in common between the best ranked genes and the training genes")
print(f"Enrichment rate : {ratio:.3f} %")

with open(f"{PPR_dir}/Enrichment/Enrichment_info_{dataset}_{process}.txt", "a") as f_out:
    f_out.write(f"There are {len(common)} GO annotations in common between the best ranked genes and the training genes\n")
    f_out.write(f"Enrichment rate : {ratio:.3f} %")


# Computing the Jaccard similarity index and Jaccard dissimilarity distance
reference = pd.read_csv(f"{PPR_dir}/Enrichment/Training_genes/relevant_GO_training_genes.csv")
pathways_reference = set()

for pathway in reference.itertuples():
    pathways_reference.add(pathway[1])

predicted = pd.read_csv(f"{PPR_dir}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv")
pathways_predicted = set()

for pathway in predicted.itertuples():
    pathways_predicted.add(pathway[1])


#Intersection and Union of two sets can also be done using & and | operators
AnB = pathways_reference.intersection(pathways_predicted)
AUB = pathways_reference.union(pathways_predicted)

JAB = float(len(AnB))/float(len(AUB))

#Computing the Jaccard similarity index and Jaccard dissimilarity distance
print(f"Jaccard similarity index J(A,B): {JAB:.3f}")
print(f"Jaccard dissimilarity distance : {(1 - JAB):.3f}")

#Writting the results in the .txt file
with open(f"{PPR_dir}/Enrichment/Enrichment_info_{dataset}_{process}.txt", "a") as f_out:
    f_out.write(f"\nJaccard similarity index : {JAB:.3f}\n")
    f_out.write(f"Jaccard dissimilarity distance : {(1 - JAB):.3f}")

### *Preparing a dedicated subfolder to run a Pubmed Search* 

In [ ]:
# Importing the Pubmed search script and aliases file to the PPR folder
if not os.path.exists(f"{PPR_dir}/Aliases.csv"):
    shutil.copyfile("../PubmedSearch/Aliases.csv", f"{PPR_dir}/Aliases.csv")

if not os.path.exists(f"{PPR_dir}/Pubmed_search_Leo_Bettoni.py"):
    shutil.copyfile("../PubmedSearch/Pubmed_search_Leo_Bettoni.py", 
                    f"{PPR_dir}/Pubmed_search_Leo_Bettoni.py")